# Baseline coverage & convergence recomputation

Refits the five focal models (BP, EI, GG, PM, FS) on the 11 baseline (`est-default`)
simulated datasets to compute two things the main simulation grid did not retain:

1. **Interval calibration**: empirical coverage of central 50% / 90% posterior intervals
   for the true turnout of every 1D and 2D demographic margin. The main grid collapsed
   `draws.parquet` to per-chain means and deleted `idata.nc`, so coverage cannot be
   recovered from the stored artifacts.
2. **Convergence diagnostics beyond mean R-hat**: rank-normalized split R-hat
   (Vehtari et al. 2021) maximum, share of parameter values above 1.01 / 1.05 / 1.1,
   and minimum bulk ESS. The grid stored only the mean R-hat across parameters.

The fitting protocol matches `run-simulations.py`: nutpie NUTS, 2 chains,
800 tuning / 500 draws, `target_accept=0.9`, refit at 0.99 when a run has more than
50 divergences (keeping the run with fewest divergences). All margin-using models
receive municipality-level margins, as in the baseline grid.

**Note**: this recomputation was executed with newer package versions than the original
grid (original: PyMC 5.28.4, PyTensor 2.38.2, nutpie 0.16.6); versions used here are
printed below.

Run headless with:
```
jupyter nbconvert --to notebook --execute --inplace coverage-rerun.ipynb --ExecutePreprocessor.timeout=-1
```
Fits are cached under `../tmp/coverage/`, so the notebook can be interrupted and re-run.

In [1]:
import os
import sys

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

import itertools as it
import json
import pathlib
from time import perf_counter

import arviz as az
import numpy as np
import pandas as pd
import nutpie
import pymc
import pytensor

import salk_turnout_models as tm

print('pymc', pymc.__version__, '| pytensor', pytensor.__version__,
      '| nutpie', nutpie.__version__, '| arviz', az.__version__)

DATA_PREFIX = pathlib.Path('../tmp/data')
OUT_PREFIX = pathlib.Path('../tmp/coverage')
OUT_PREFIX.mkdir(parents=True, exist_ok=True)

INPUT_COLS = sorted(['age_group', 'gender', 'education', 'municipality', 'nationality'])
MARGINS_1D = [[c] for c in INPUT_COLS]
MARGINS_2D = [list(p) for p in it.combinations(INPUT_COLS, 2)]
ALL_MARGINS = [[]] + MARGINS_1D + MARGINS_2D
FOCAL_MODELS = ['1_bp', '2_ei', '3_gg', '4_pm', '5_fs']

pymc 6.0.1 | pytensor 3.0.7 | nutpie 0.16.10 | arviz 1.1.0


In [2]:
data_list = json.load(open(DATA_PREFIX / 'data_list.json'))
default_datasets = [name for name, _ in data_list['est-default']][:11]
print(len(default_datasets), 'baseline datasets')


def model_configs(data_name):
    d = DATA_PREFIX / data_name
    base = {
        'outcome_col': 'voting_intent',
        'population': '../data/census.csv',
        'input_cols': INPUT_COLS,
    }
    survey = str(d / 'estonia_selection.csv')
    margin = [str(d / 'estonia_municipality_margins.csv')]
    return [
        base | {'name': '1_bp', 'model_type': 'BP', 'centered': False, 'survey': survey},
        base | {'name': '2_ei', 'model_type': 'EI', 'centered': True, 'margin': margin},
        base | {'name': '3_gg', 'model_type': 'GG', 'centered': False, 'survey': survey, 'margin': margin},
        base | {'name': '4_pm', 'model_type': 'PM', 'centered': True, 'survey': survey, 'margin': margin},
        base | {'name': '5_fs', 'model_type': 'FS', 'centered': True, 'survey': survey, 'margin': margin},
    ]

11 baseline datasets


In [3]:
def margin_draws(draws_df):
    """Per-draw turnout rates for the topline and all 1D/2D margins."""
    frames = []
    for cols in ALL_MARGINS:
        keys = ['chain', 'draw'] + cols
        g = draws_df.groupby(keys, observed=False)[['N', 'N_census']].sum().reset_index()
        g['turnout'] = g['N'] / g['N_census']
        g['margin'] = '*'.join(cols) if cols else 'topline'
        g['category'] = g[cols].astype(str).agg('|'.join, axis=1) if cols else 'all'
        frames.append(g[['margin', 'category', 'chain', 'draw', 'turnout']])
    return pd.concat(frames, ignore_index=True)


def margin_truth(pop_df):
    """True turnout rates per margin category from the simulated population."""
    yes = pop_df['voting_intent'] == 'Yes'
    rows = []
    for cols in ALL_MARGINS:
        if cols:
            tot = pop_df.groupby(cols, observed=False)['N'].sum()
            vot = pop_df[yes].groupby(cols, observed=False)['N'].sum()
            t = (vot / tot).reset_index(name='true_turnout')
            t['category'] = t[cols].astype(str).agg('|'.join, axis=1)
        else:
            t = pd.DataFrame({
                'category': ['all'],
                'true_turnout': [pop_df.loc[yes, 'N'].sum() / pop_df['N'].sum()],
            })
        t['margin'] = '*'.join(cols) if cols else 'topline'
        rows.append(t[['margin', 'category', 'true_turnout']])
    return pd.concat(rows, ignore_index=True)


def rhat_diagnostics(idata):
    """Rank-normalized split R-hat (arviz default; Vehtari et al. 2021) and bulk ESS."""
    rhat = az.rhat(idata)
    vals = np.concatenate([np.asarray(v.values, float).ravel() for v in rhat.data_vars.values()])
    vals = vals[np.isfinite(vals)]
    ess = az.ess(idata)
    ess_vals = np.concatenate([np.asarray(v.values, float).ravel() for v in ess.data_vars.values()])
    ess_vals = ess_vals[np.isfinite(ess_vals)]
    return {
        'mean_rhat': float(vals.mean()),
        'max_rhat': float(vals.max()),
        'frac_rhat_gt_101': float((vals > 1.01).mean()),
        'frac_rhat_gt_105': float((vals > 1.05).mean()),
        'frac_rhat_gt_110': float((vals > 1.1).mean()),
        'n_rhat_values': int(vals.size),
        'min_ess_bulk': float(ess_vals.min()),
    }

In [4]:
def fit_one(desc, data_name):
    out_dir = OUT_PREFIX / data_name / desc['name']
    diag_path = out_dir / 'diagnostics.json'
    if diag_path.exists():
        return json.load(open(diag_path))
    out_dir.mkdir(parents=True, exist_ok=True)

    # Same protocol as run-simulations.py: refit at 0.99 if >50 divergences,
    # keep the run with fewest divergences.
    best = None
    for target_accept in (0.9, 0.99):
        t0 = perf_counter()
        result = tm.run_model(desc, sample_kwargs={'target_accept': target_accept})
        fit_time = perf_counter() - t0
        divergences = int(result['idata'].sample_stats.diverging.sum().item())
        run = {
            'result': result,
            'divergences': divergences,
            'target_accept': target_accept,
            'fit_time': fit_time,
        }
        if best is None or divergences < best['divergences']:
            best = run
        if divergences <= 50:
            break

    diag = rhat_diagnostics(best['result']['idata'])
    diag |= {
        'model': desc['name'],
        'data_id': data_name,
        'divergences': best['divergences'],
        'target_accept': best['target_accept'],
        'fit_time': best['fit_time'],
    }
    margin_draws(best['result']['draws']).to_parquet(out_dir / 'margin_draws.parquet', index=False)
    with open(diag_path, 'w') as f:
        json.dump(diag, f, indent=2)
    return diag


diags = []
for data_name in default_datasets:
    for desc in model_configs(data_name):
        print(data_name, desc['name'], flush=True)
        diags.append(fit_one(desc, data_name))

diag_df = pd.DataFrame(diags)
diag_df

est-default-2bb62265c58cf13eeee333684c133bed 1_bp
est-default-2bb62265c58cf13eeee333684c133bed 2_ei
est-default-2bb62265c58cf13eeee333684c133bed 3_gg
est-default-2bb62265c58cf13eeee333684c133bed 4_pm
est-default-2bb62265c58cf13eeee333684c133bed 5_fs
est-default-06a86ff0c4d6aeb789b273de20828128 1_bp
est-default-06a86ff0c4d6aeb789b273de20828128 2_ei
est-default-06a86ff0c4d6aeb789b273de20828128 3_gg
est-default-06a86ff0c4d6aeb789b273de20828128 4_pm
est-default-06a86ff0c4d6aeb789b273de20828128 5_fs
est-default-6461febcf38c6d2a26eb409029a39916 1_bp
est-default-6461febcf38c6d2a26eb409029a39916 2_ei
est-default-6461febcf38c6d2a26eb409029a39916 3_gg
est-default-6461febcf38c6d2a26eb409029a39916 4_pm
est-default-6461febcf38c6d2a26eb409029a39916 5_fs
est-default-9bf91b6d469341eece9930cd806e7bb5 1_bp
est-default-9bf91b6d469341eece9930cd806e7bb5 2_ei
est-default-9bf91b6d469341eece9930cd806e7bb5 3_gg
est-default-9bf91b6d469341eece9930cd806e7bb5 4_pm
est-default-9bf91b6d469341eece9930cd806e7bb5 5_fs


,mean_rhat,max_rhat,frac_rhat_gt_101,frac_rhat_gt_105,frac_rhat_gt_110,n_rhat_values,min_ess_bulk,model,data_id,divergences,target_accept,fit_time
0,1.001606,1.015547,0.008646,0.000000,0.000000,2082,122.561009,1_bp,est-default-2bb62265c58cf13eeee333684c133bed,0,0.90,34.925183
1,1.010831,1.060232,0.438100,0.000960,0.000000,2084,39.995006,2_ei,est-default-2bb62265c58cf13eeee333684c133bed,4,0.90,175.119498
2,1.001543,1.016794,0.008646,0.000000,0.000000,2082,79.399394,3_gg,est-default-2bb62265c58cf13eeee333684c133bed,0,0.90,124.067224
3,1.009991,1.045588,0.407589,0.000000,0.000000,4085,47.680105,4_pm,est-default-2bb62265c58cf13eeee333684c133bed,4,0.90,88.887926
4,1.012936,1.117241,0.397986,0.043017,0.005144,9136,11.762777,5_fs,est-default-2bb62265c58cf13eeee333684c133bed,14,0.90,143.250443
5,1.001195,1.014924,0.003842,0.000000,0.000000,2082,120.152080,1_bp,est-default-06a86ff0c4d6aeb789b273de20828128,0,0.90,46.018853
6,1.033136,1.149331,0.891075,0.194818,0.002879,2084,11.178541,2_ei,est-default-06a86ff0c4d6aeb789b273de20828128,4,0.90,191.175533
7,1.001133,1.050503,0.010086,0.000480,0.000000,2082,89.156684,3_gg,est-default-06a86ff0c4d6aeb789b273de20828128,0,0.90,103.827938
8,1.009038,1.050635,0.341004,0.000245,0.000000,4085,33.382262,4_pm,est-default-06a86ff0c4d6aeb789b273de20828128,0,0.90,116.816820
9,1.025788,1.139546,0.491462,0.209391,0.058778,9136,10.015035,5_fs,est-default-06a86ff0c4d6aeb789b273de20828128,6,0.90,125.218291


## Convergence diagnostics summary

The main grid reported only the mean R-hat across parameters; this table relates that
to the stricter standard diagnostics (max rank-normalized R-hat, share of parameter
values above 1.01) for the baseline scenario.

In [5]:
diag_summary = diag_df.groupby('model').agg(
    mean_rhat_avg=('mean_rhat', 'mean'),
    max_rhat_median=('max_rhat', 'median'),
    max_rhat_worst=('max_rhat', 'max'),
    frac_gt_101_avg=('frac_rhat_gt_101', 'mean'),
    min_ess_bulk_median=('min_ess_bulk', 'median'),
    min_ess_bulk_worst=('min_ess_bulk', 'min'),
    divergences_total=('divergences', 'sum'),
    fit_time_median=('fit_time', 'median'),
)
print(diag_summary.round(4).to_string())

       mean_rhat_avg  max_rhat_median  max_rhat_worst  frac_gt_101_avg  min_ess_bulk_median  min_ess_bulk_worst  divergences_total  fit_time_median
model                                                                                                                                              
1_bp          1.0015           1.0213          1.0380           0.0100             120.4684             86.4837                  2          32.6264
2_ei          1.0158           1.0602          1.1493           0.6051              41.9264             11.1785                 30         157.1895
3_gg          1.0017           1.0204          1.0505           0.0131             125.8785             79.3994                  0          73.5137
4_pm          1.0054           1.0199          1.0506           0.1601             113.0433             33.3823                 50          76.3760
5_fs          1.0278           1.1172          2.2358           0.4669              12.9987              2.5517 

## Interval calibration (coverage)

Empirical coverage of central 50% and 90% posterior intervals for true margin turnout,
pooled over margin categories and the 11 baseline datasets. `mdim` is the margin
dimensionality (0 = topline, 1 = one-way, 2 = two-way). `width90` is the average width
of the 90% interval — narrow intervals with nominal coverage are the goal.

In [6]:
rows = []
for data_name in default_datasets:
    pop = pd.read_parquet(DATA_PREFIX / data_name / 'population.parquet')
    truth = margin_truth(pop)
    for model in FOCAL_MODELS:
        md = pd.read_parquet(OUT_PREFIX / data_name / model / 'margin_draws.parquet')
        q = md.groupby(['margin', 'category'])['turnout'].quantile([0.05, 0.25, 0.75, 0.95]).unstack()
        q.columns = [f'q{int(c * 100):02d}' for c in q.columns]
        q = q.reset_index().merge(truth, on=['margin', 'category'])
        q = q.dropna(subset=['true_turnout', 'q05'])
        q['model'] = model
        q['data_id'] = data_name
        q['mdim'] = q['margin'].map(lambda m: 0 if m == 'topline' else m.count('*') + 1)
        rows.append(q)

cov_df = pd.concat(rows, ignore_index=True)
cov_df['cover50'] = (cov_df.true_turnout >= cov_df.q25) & (cov_df.true_turnout <= cov_df.q75)
cov_df['cover90'] = (cov_df.true_turnout >= cov_df.q05) & (cov_df.true_turnout <= cov_df.q95)
cov_df['width90'] = cov_df.q95 - cov_df.q05

cov_summary = cov_df.groupby(['model', 'mdim'])[['cover50', 'cover90', 'width90']].mean()
print('Empirical coverage of central posterior intervals (nominal 0.50 / 0.90):')
print(cov_summary.round(3).to_string())

Empirical coverage of central posterior intervals (nominal 0.50 / 0.90):
            cover50  cover90  width90
model mdim                           
1_bp  0       0.000    0.000    0.049
      1       0.007    0.029    0.133
      2       0.009    0.055    0.168
2_ei  0       1.000    1.000    0.002
      1       0.835    0.974    0.192
      2       0.613    0.928    0.539
3_gg  0       0.818    1.000    0.080
      1       0.737    0.928    0.187
      2       0.626    0.922    0.232
4_pm  0       1.000    1.000    0.002
      1       0.746    0.914    0.039
      2       0.354    0.771    0.097
5_fs  0       1.000    1.000    0.002
      1       0.782    0.959    0.039
      2       0.469    0.897    0.095


In [7]:
# Paper-ready summary lines
print('Coverage (1D margins, nominal 50%/90%):')
for model in FOCAL_MODELS:
    sub = cov_df[(cov_df.model == model) & (cov_df['mdim'] == 1)]
    print(f"  {model}: {sub.cover50.mean():.2f} / {sub.cover90.mean():.2f}"
          f" (mean 90% width {sub.width90.mean():.3f})")
print('Coverage (2D margins, nominal 50%/90%):')
for model in FOCAL_MODELS:
    sub = cov_df[(cov_df.model == model) & (cov_df['mdim'] == 2)]
    print(f"  {model}: {sub.cover50.mean():.2f} / {sub.cover90.mean():.2f}"
          f" (mean 90% width {sub.width90.mean():.3f})")

Coverage (1D margins, nominal 50%/90%):
  1_bp: 0.01 / 0.03 (mean 90% width 0.133)
  2_ei: 0.83 / 0.97 (mean 90% width 0.192)
  3_gg: 0.74 / 0.93 (mean 90% width 0.187)
  4_pm: 0.75 / 0.91 (mean 90% width 0.039)
  5_fs: 0.78 / 0.96 (mean 90% width 0.039)
Coverage (2D margins, nominal 50%/90%):
  1_bp: 0.01 / 0.06 (mean 90% width 0.168)
  2_ei: 0.61 / 0.93 (mean 90% width 0.539)
  3_gg: 0.63 / 0.92 (mean 90% width 0.232)
  4_pm: 0.35 / 0.77 (mean 90% width 0.097)
  5_fs: 0.47 / 0.90 (mean 90% width 0.095)
